In [ ]:
!pip -q install -U huggingface_hub datasets pyarrow pandas pillow scikit-learn

In [ ]:
from pathlib import Path
import os, json, random, hashlib, shutil, subprocess, zipfile, io, sys, platform
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image, ImageOps

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

from google.colab import drive
drive.mount('/content/drive')

PROJECT = Path('/content/drive/MyDrive/TwinCheck')
RAW = PROJECT / 'data' / 'raw' / 'vision'
PROCESSED = PROJECT / 'data' / 'processed' / 'vision'
REPORTS = PROJECT / 'reports' / 'F1'
VERITE_RAW = RAW / 'verite'
GENIMAGE_RAW = RAW / 'genimage'
VERITE_OUT = PROCESSED / 'verite'
GENIMAGE_OUT = PROCESSED / 'genimage'

for folder in [RAW, PROCESSED, REPORTS, VERITE_RAW, GENIMAGE_RAW, VERITE_OUT, GENIMAGE_OUT]:
    folder.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT)
print('Created/check folders:', len([RAW, PROCESSED, REPORTS, VERITE_RAW, GENIMAGE_RAW, VERITE_OUT, GENIMAGE_OUT]))

In [ ]:
from huggingface_hub import login, HfApi, snapshot_download

login()
api = HfApi()

runtime_info = {
    'python': sys.version,
    'platform': platform.platform(),
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'pillow': Image.__version__,
}
print(json.dumps(runtime_info, indent=2))

In [ ]:
VERITE_GITHUB = 'https://github.com/stevejpapad/image-text-verification.git'
VERITE_REPO_LOCAL = Path('/content/twincheck_f1_verite_repo')
if VERITE_REPO_LOCAL.exists():
    shutil.rmtree(VERITE_REPO_LOCAL)
subprocess.run(['git', 'clone', '--depth', '1', VERITE_GITHUB, str(VERITE_REPO_LOCAL)], check=True)
verite_git_sha = subprocess.check_output(['git', '-C', str(VERITE_REPO_LOCAL), 'rev-parse', 'HEAD'], text=True).strip()

def find_named_file(root, filename):
    matches = list(Path(root).rglob(filename))
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one {filename}; found {len(matches)}: {matches[:5]}')
    return matches[0]

VERITE_CSV = find_named_file(VERITE_REPO_LOCAL, 'VERITE.csv')
VERITE_ARTICLES_CSV = find_named_file(VERITE_REPO_LOCAL, 'VERITE_articles.csv')
verite_meta = pd.read_csv(VERITE_CSV)
verite_articles = pd.read_csv(VERITE_ARTICLES_CSV)
verite_meta = verite_meta.loc[:, [c for c in verite_meta.columns if not str(c).startswith('Unnamed:')]]
verite_articles = verite_articles.loc[:, [c for c in verite_articles.columns if not str(c).startswith('Unnamed:')]]

required_cols = {'image_path', 'label'}
missing_cols = required_cols - set(verite_meta.columns)
if missing_cols:
    raise ValueError(f'Official VERITE.csv is missing required columns: {missing_cols}')
expected_verite_labels={'true','miscaptioned','out-of-context'}
observed_verite_labels={str(x).strip().lower() for x in verite_meta['label'].dropna().unique()}
if not observed_verite_labels.issubset(expected_verite_labels):
    raise ValueError(f'Unexpected VERITE label values: {observed_verite_labels}')
print('Official metadata file:', VERITE_CSV)
print('VERITE rows:', len(verite_meta), 'article metadata rows:', len(verite_articles))
print('Columns:', verite_meta.columns.tolist())
print('Label counts:', verite_meta['label'].value_counts(dropna=False).to_dict())
print('Repository commit:', verite_git_sha)
print(verite_meta.head().to_string(index=False))

In [ ]:
VERITE_HF_ID = 'stefpapad/VERITE'
verite_hf_info = api.dataset_info(VERITE_HF_ID)
verite_hf_revision = verite_hf_info.sha
VERITE_HF_LOCAL = VERITE_RAW / 'hf_snapshot'
VERITE_HF_LOCAL.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id=VERITE_HF_ID, repo_type='dataset', revision=verite_hf_revision,
    local_dir=str(VERITE_HF_LOCAL)
)

VERITE_ZIP = VERITE_HF_LOCAL / 'images.zip'
IMAGE_ROOT = VERITE_RAW / 'images_extracted'
IMAGE_ROOT.mkdir(parents=True, exist_ok=True)
if not VERITE_ZIP.is_file():
    raise FileNotFoundError(f'Expected archive not found: {VERITE_ZIP}')

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}
with zipfile.ZipFile(VERITE_ZIP) as zf:
    image_members = [m for m in zf.infolist() if not m.is_dir() and Path(m.filename).suffix.lower() in IMAGE_EXTS]
    if not image_members:
        raise RuntimeError('No supported image files found in VERITE images.zip')
    root_resolved = IMAGE_ROOT.resolve()
    for member in image_members:
        target = (IMAGE_ROOT / member.filename).resolve()
        if not target.is_relative_to(root_resolved):
            raise RuntimeError(f'Unsafe ZIP member path: {member.filename}')
        if not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(member) as src, target.open('wb') as dst:
                shutil.copyfileobj(src, dst)

verite_image_files = [
    p for p in IMAGE_ROOT.rglob('*')
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
]
print('VERITE HF revision:', verite_hf_revision)
print('Snapshot directory:', VERITE_HF_LOCAL)
print('Archive:', VERITE_ZIP, f'({VERITE_ZIP.stat().st_size/1024/1024:.2f} MB)')
print('Extracted image root:', IMAGE_ROOT)
print('ZIP image entries:', len(image_members), '| extracted images:', len(verite_image_files))
if not verite_image_files:
    raise RuntimeError('No extracted VERITE images found.')

In [ ]:
def norm_rel(s):
    return str(s).replace('\\\\', '/').strip().lstrip('./')

rel_lookup = {}
base_lookup = {}
for p in verite_image_files:
    rel = norm_rel(p.relative_to(IMAGE_ROOT))
    rel_lookup.setdefault(rel, []).append(p)
    base_lookup.setdefault(p.name, []).append(p)

ambiguous_basename_rows = []
missing_verite_rows = []
resolved_paths = []
for source_path in verite_meta['image_path'].astype(str):
    key = norm_rel(source_path)
    candidates = rel_lookup.get(key, [])
    if not candidates:
        candidates = [p for rel, paths in rel_lookup.items() if rel.endswith('/' + key) or key.endswith('/' + rel) for p in paths]
    candidates = list(dict.fromkeys(candidates))
    if len(candidates) == 1:
        resolved_paths.append(str(candidates[0]))
        continue
    if len(candidates) > 1:
        ambiguous_basename_rows.append({'image_path': source_path, 'reason': 'relative path ambiguous', 'candidate_paths': [str(p) for p in candidates]})
        resolved_paths.append('')
        continue
    candidates = base_lookup.get(Path(key).name, [])
    if len(candidates) == 1:
        resolved_paths.append(str(candidates[0]))
    elif len(candidates) > 1:
        ambiguous_basename_rows.append({'image_path': source_path, 'reason': 'basename ambiguous', 'candidate_paths': [str(p) for p in candidates]})
        resolved_paths.append('')
    else:
        missing_verite_rows.append(source_path)
        resolved_paths.append('')

verite_check = verite_meta.copy()
verite_check['resolved_image_path'] = resolved_paths
verite_check['image_exists'] = verite_check['resolved_image_path'].map(lambda s: bool(s) and Path(s).is_file())
print('Metadata rows:', len(verite_check))
print('Resolved:', int(verite_check.image_exists.sum()))
print('Missing:', int((~verite_check.image_exists).sum()))
print('Ambiguous:', len(ambiguous_basename_rows))
print('Labels:', verite_check.label.value_counts(dropna=False).to_dict())

verite_match_report = {
    'metadata_rows': int(len(verite_check)),
    'resolved_rows': int(verite_check.image_exists.sum()),
    'missing_rows': int((~verite_check.image_exists).sum()),
    'ambiguous_matches': ambiguous_basename_rows,
    'missing_image_paths': missing_verite_rows,
}
with open(REPORTS/'verite_image_match_report.json', 'w') as f:
    json.dump(verite_match_report, f, indent=2)
if ambiguous_basename_rows:
    print('WARNING: Ambiguous matches were not silently resolved. Inspect report.')
if not verite_check.image_exists.any():
    raise RuntimeError('No VERITE metadata rows resolved to images; stop and inspect image-path structure.')

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

valid_verite = verite_check[verite_check.image_exists].copy()

image_info = {}
for path in valid_verite['resolved_image_path'].unique():
    try:
        with Image.open(path) as im:
            im.verify()
        raw_hash = hashlib.sha256(Path(path).read_bytes()).hexdigest()
        image_info[path] = {'readable': True, 'group_id': raw_hash}
    except Exception as exc:
        image_info[path] = {'readable': False, 'group_id': None, 'error': repr(exc)}

valid_verite['image_readable'] = valid_verite.resolved_image_path.map(lambda p: image_info[p]['readable'])
valid_verite = valid_verite[valid_verite.image_readable].copy()
valid_verite['group_id'] = valid_verite.resolved_image_path.map(lambda p: image_info[p]['group_id'])
if valid_verite.empty:
    raise RuntimeError('No readable VERITE rows remain after validation.')

gss_a = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
idx_trainval, idx_test = next(gss_a.split(valid_verite, groups=valid_verite.group_id))
trainval = valid_verite.iloc[idx_trainval].copy()
test = valid_verite.iloc[idx_test].copy()
gss_b = GroupShuffleSplit(n_splits=1, test_size=0.15/0.85, random_state=SEED)
idx_train, idx_val = next(gss_b.split(trainval, groups=trainval.group_id))
train = trainval.iloc[idx_train].copy()
val = trainval.iloc[idx_val].copy()

split_dfs = {'train': train, 'validation': val, 'test': test}
group_sets = {k: set(v.group_id) for k, v in split_dfs.items()}
assert not (group_sets['train'] & group_sets['validation'])
assert not (group_sets['train'] & group_sets['test'])
assert not (group_sets['validation'] & group_sets['test'])

NORMALIZED_SIZE = (256, 256)
def standardize_image(source_path, destination_path):
    destination_path = Path(destination_path)
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    with Image.open(source_path) as im:
        rgb = ImageOps.exif_transpose(im).convert('RGB')
        rgb = rgb.resize(NORMALIZED_SIZE, Image.Resampling.LANCZOS)
        rgb.save(destination_path, format='JPEG', quality=95, subsampling=0, optimize=True)
    return destination_path

verite_export = []
for split_name, split_df in split_dfs.items():
    for row in split_df.to_dict('records'):
        source = Path(row['resolved_image_path'])
        image_key = row['group_id'][:20]
        raw_dest = VERITE_OUT/'raw_images'/split_name/(image_key+source.suffix.lower())
        norm_dest = VERITE_OUT/'normalized_images'/split_name/(image_key+'.jpg')
        raw_dest.parent.mkdir(parents=True, exist_ok=True)
        if not raw_dest.exists():
            shutil.copy2(source, raw_dest)
        standardize_image(source, norm_dest)
        record = {k: v for k, v in row.items() if k not in ['resolved_image_path', 'image_exists', 'image_readable']}
        record.update({'split': split_name, 'group_id': row['group_id'],
                      'source_image_path': str(source), 'raw_export_path': str(raw_dest),
                      'normalized_image_path': str(norm_dest),
                      'raw_sha256': row['group_id'],
                      'raw_md5': hashlib.md5(raw_dest.read_bytes()).hexdigest(),
                      'normalized_md5': hashlib.md5(norm_dest.read_bytes()).hexdigest()})
        verite_export.append(record)

verite_manifest = pd.DataFrame(verite_export)
for split_name in split_dfs:
    split_manifest = verite_manifest[verite_manifest.split == split_name]
    split_manifest.to_json(VERITE_OUT/f'verite_{split_name}.jsonl', orient='records', lines=True, force_ascii=False)
verite_manifest.to_json(VERITE_OUT/'verite.jsonl', orient='records', lines=True, force_ascii=False)
verite_manifest.to_csv(VERITE_OUT/'verite_manifest.csv', index=False)
print('VERITE split records:', verite_manifest.groupby('split').size().to_dict())
print('VERITE split unique image groups:', verite_manifest.groupby('split').group_id.nunique().to_dict())
print('VERITE normalized images created:', len(verite_manifest))

In [ ]:
REDDOT_REPO_URL = 'https://github.com/stevejpapad/relevant-evidence-detection.git'
REDDOT_LOCAL = Path('/content/twincheck_f1_red_dot')
red_dot_report = {'attempted': True, 'repo_url': REDDOT_REPO_URL, 'available': False,
                  'json_files_found': 0, 'empty_files': 0, 'nonempty_files': 0, 'invalid_files': 0,
                  'evidence_relevance_verified': False, 'error': None}
try:
    if not REDDOT_LOCAL.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REDDOT_REPO_URL, str(REDDOT_LOCAL)],
                       check=True, capture_output=True, text=True)
    else:
        if not (REDDOT_LOCAL/'.git').exists():
            raise RuntimeError(f'Existing path is not a Git clone: {REDDOT_LOCAL}')
    red_dot_report['available'] = True
    red_dot_report['commit'] = subprocess.check_output(
        ['git', '-C', str(REDDOT_LOCAL), 'rev-parse', 'HEAD'], text=True).strip()
    candidates = [p for p in REDDOT_LOCAL.rglob('*.json')
                  if 'verite' in str(p).lower() or 'evidence' in p.name.lower()]
    empty, nonempty, invalid = [], [], []
    for p in candidates:
        try:
            obj = json.loads(p.read_text(encoding='utf-8'))
            (empty if obj in ([], {}, None) else nonempty).append(str(p.relative_to(REDDOT_LOCAL)))
        except Exception as e:
            invalid.append({'path': str(p.relative_to(REDDOT_LOCAL)), 'error': repr(e)})
    red_dot_report.update({'json_files_found': len(candidates), 'empty_files': len(empty),
                           'nonempty_files': len(nonempty), 'invalid_files': len(invalid),
                           'empty_file_paths': empty, 'nonempty_file_paths': nonempty,
                           'invalid_file_details': invalid})
except Exception as exc:
    red_dot_report['error'] = repr(exc)
    print('RED-DOT check could not complete; recorded as unavailable, not PASS:', repr(exc))
print(json.dumps(red_dot_report, indent=2))
with open(REPORTS/'red_dot_evidence_availability.json', 'w') as f:
    json.dump(red_dot_report, f, indent=2)
print('Note: non-empty evidence files do not establish evidence relevance or correctness.')

In [ ]:
from datasets import load_dataset
from huggingface_hub import HfApi

GENIMAGE_REPO = 'nebula/GenImage-arrow'
GENIMAGE_CONFIGS = {'biggan_train':'biggan-train', 'sd14_train':'sd14-train'}
genimage_info = api.dataset_info(GENIMAGE_REPO)
GENIMAGE_REVISION = genimage_info.sha
repo_files = api.list_repo_files(repo_id=GENIMAGE_REPO, repo_type='dataset', revision=GENIMAGE_REVISION)
print('GenImage pinned revision:', GENIMAGE_REVISION)
print('Repository file count:', len(repo_files))
print('Requested configs:', GENIMAGE_CONFIGS)
print('Dataset card/source reference: https://huggingface.co/datasets/nebula/GenImage-arrow')

for config in GENIMAGE_CONFIGS.values():
    ds = load_dataset(GENIMAGE_REPO, config, split='train', streaming=True, revision=GENIMAGE_REVISION)
    print('Config:',config,'columns:',ds.column_names)
    first = next(iter(ds))
    print('First row keys:',list(first.keys()))
    print('Label sample:',first.get('label'),'generator sample:',first.get('generator'),
          'image_path sample:',first.get('image_path'))
    if not {'image','label'}.issubset(set(ds.column_names)):
        raise RuntimeError(f'{config} missing required columns image/label: {ds.column_names}')

In [ ]:
from io import BytesIO
from pathlib import Path
import hashlib
import numpy as np
from PIL import Image

TARGETS = {
    'real': 6500,
    'biggan_fake': 6500,
    'sd14_fake': 1500
}

FINAL_TARGETS = {
    'real': 6000,
    'biggan_fake': 6000,
    'sd14_fake': 1000
}

BUFFER_SIZE = 10000

def get_image_bytes(image_value, image_path=''):

    if isinstance(image_value, dict):

        raw = image_value.get('bytes')

        if raw is not None:
            return raw, 'original_bytes'

        path = image_value.get('path')

        if path and Path(path).is_file():
            return Path(path).read_bytes(), 'local_file_bytes'

    elif isinstance(image_value, (bytes, bytearray)):
        return bytes(image_value), 'original_bytes'

    elif isinstance(image_value, Image.Image):

        buffer = BytesIO()

        image_value.convert('RGB').save(
            buffer,
            format='PNG'
        )

        return buffer.getvalue(), 'reencoded_from_decoded_pil'

    raise TypeError(
        f'Unsupported image field type: {type(image_value)}'
    )

def label_to_int(value):

    if isinstance(value, (int, np.integer)):
        value = int(value)

        if value in (0, 1):
            return value

    if isinstance(value, str):

        value_normalized = value.strip().lower()

        if value_normalized in {'real', 'nature', '0'}:
            return 0

        if value_normalized in {
            'fake', 'ai', 'generated', '1'
            }:
            return 1

    raise ValueError(
        f'Unexpected/unmapped label value: {value!r}'
    )

def decode_generator(value, generator_feature):

    if value is None:
        return None

    if (
        isinstance(value, (int, np.integer))
        and generator_feature is not None
        and hasattr(generator_feature, 'int2str')
    ):

        try:
            return generator_feature.int2str(int(value))

        except (ValueError, IndexError):
            return f'UNKNOWN_GENERATOR_ID:{value}'

    return str(value)

def normalize_generator_name(value):

    import re

    return re.sub(
        r'[^a-z0-9]+',
        '',
        str(value).strip().lower()
    )

def generator_matches(decoded_value, expected):

    if decoded_value is None:
        return False

    actual = normalize_generator_name(decoded_value)
    expected_normalized = normalize_generator_name(expected)

    aliases = {
        'BigGAN': {
            'biggan'
        },

        'Stable Diffusion v1.4': {
            'stablediffusionv14',
            'stablediffusionv1.4',
            'sd14',
            'sdv14',
            'stable_diffusion_v_1_4'
        }
    }

    if expected in aliases:

        normalized_aliases = {
            normalize_generator_name(alias)
            for alias in aliases[expected]
        }

        return actual in normalized_aliases

    return actual == expected_normalized

def collect_unique_candidates(
    config,
    wanted_label,
    expected_source_generator,
    output_generator,
    needed,
    seed
):

    print(f'\nCollecting candidates: {config}')
    print(f'Required label: {wanted_label}')
    print(f'Required generator: {expected_source_generator}')
    print(f'Required unique candidates: {needed}')

    ds = load_dataset(
        GENIMAGE_REPO,
        config,
        split='train',
        streaming=True,
        revision=GENIMAGE_REVISION
    )

    generator_feature = ds.features.get('generator')

    if generator_feature is not None:

        print(
            'Generator class names:',
            getattr(generator_feature, 'names', None)
        )

    ds = ds.shuffle(
        seed=seed,
        buffer_size=BUFFER_SIZE
    )

    records = []
    seen_hashes = set()

    observed_labels = set()
    observed_generators = set()

    skipped_bad_images = 0
    skipped_duplicate_images = 0
    skipped_generator_mismatch = 0
    missing_generator_metadata = 0

    for row in ds:

        label = label_to_int(row['label'])

        observed_labels.add(label)

        if label != wanted_label:
            continue

        raw_generator_value = row.get('generator')

        decoded_generator_value = decode_generator(
            raw_generator_value,
            generator_feature
        )

        if decoded_generator_value is not None:
            observed_generators.add(decoded_generator_value)

        if label == 1:

            if decoded_generator_value is None:

                missing_generator_metadata += 1

                continue

            if not generator_matches(
                decoded_generator_value,
                expected_source_generator
            ):

                skipped_generator_mismatch += 1

                continue

        try:

            raw, byte_status = get_image_bytes(
                row['image'],
                row.get('image_path', '')
            )

        except Exception as e:

            skipped_bad_images += 1

            if skipped_bad_images <= 5:

                print(
                    'Skipping image byte extraction failure:',
                    row.get('image_path', ''),
                    repr(e)
                )

            continue

        digest = hashlib.md5(raw).hexdigest()

        if digest in seen_hashes:

            skipped_duplicate_images += 1

            continue

        try:

            with Image.open(BytesIO(raw)) as im:

                im.verify()

            with Image.open(BytesIO(raw)) as im:

                width, height = im.size
                image_format = im.format or 'unknown'
                mode = im.mode

        except Exception as e:

            skipped_bad_images += 1

            if skipped_bad_images <= 5:

                print(
                    'Skipping unreadable image:',
                    row.get('image_path', ''),
                    repr(e)
                )

            continue

        seen_hashes.add(digest)

        records.append({

            'image_bytes': raw,

            'md5': digest,

            'label': label,

            'class_name': (
                'real' if label == 0 else 'fake'
            ),

            'generator': output_generator,

            'source_config': config,

            'source_revision': GENIMAGE_REVISION,

            'image_path': row.get('image_path', ''),

            'source_generator_metadata': decoded_generator_value,

            'source_generator_metadata_raw': raw_generator_value,

            'source_byte_status': byte_status,

            'original_format': image_format,

            'original_width': int(width),

            'original_height': int(height),

            'original_mode': mode

        })

        if len(records) >= needed:
            break

    print('\nCollection report:')
    print('  Collected:', len(records))
    print('  Observed labels:', sorted(observed_labels))
    print('  Observed generators:', sorted(observed_generators))
    print('  Skipped duplicate images:', skipped_duplicate_images)
    print('  Skipped invalid images:', skipped_bad_images)
    print('  Skipped generator mismatches:', skipped_generator_mismatch)
    print('  Missing generator metadata:', missing_generator_metadata)

    if len(records) < needed:

        raise RuntimeError(
            f'\nDataset collection FAILED for {config}.\n'
            f'Required: {needed}\n'
            f'Collected: {len(records)}\n'
            f'Observed labels: {sorted(observed_labels)}\n'
            f'Observed generators: {sorted(observed_generators)}\n'
            f'Generator mismatches: {skipped_generator_mismatch}\n'
            f'Missing generator metadata: {missing_generator_metadata}\n'
            f'Invalid images: {skipped_bad_images}\n'
            f'Duplicate images: {skipped_duplicate_images}\n'
            'Do not silently reduce target counts.'
        )

    print(
        f'\nSUCCESS: {config} collected '
        f'{len(records)} unique candidates.'
    )

    return records

real_candidates = collect_unique_candidates(
    config='biggan-train',
    wanted_label=0,
    expected_source_generator='BigGAN',
    output_generator='real',
    needed=TARGETS['real'],
    seed=SEED
)

biggan_candidates = collect_unique_candidates(
    config='biggan-train',
    wanted_label=1,
    expected_source_generator='BigGAN',
    output_generator='BigGAN',
    needed=TARGETS['biggan_fake'],
    seed=SEED + 1
)

In [ ]:
BUFFER_SIZE = 500

sd14_candidates = collect_unique_candidates(
    config='sd14-train',
    wanted_label=1,
    expected_source_generator='Stable Diffusion v1.4',
    output_generator='Stable Diffusion v1.4',
    needed=TARGETS['sd14_fake'],
    seed=SEED + 2
)

print('\n' + '=' * 60)
print('CANDIDATE COLLECTION SUMMARY')
print('=' * 60)

print('Real candidates:', len(real_candidates))
print('BigGAN fake candidates:', len(biggan_candidates))
print('Stable Diffusion v1.4 fake candidates:', len(sd14_candidates))

assert len(real_candidates) == TARGETS['real']
assert len(biggan_candidates) == TARGETS['biggan_fake']
assert len(sd14_candidates) == TARGETS['sd14_fake']

print('\nAll candidate target counts satisfied.')

In [ ]:
real_hashes = set(x['md5'] for x in real_candidates)

biggan_candidates = [
    x for x in biggan_candidates
    if x['md5'] not in real_hashes
]

biggan_hashes = set(x['md5'] for x in biggan_candidates)

sd14_candidates = [
    x for x in sd14_candidates
    if x['md5'] not in real_hashes
    and x['md5'] not in biggan_hashes
]

if (
    len(real_candidates) < FINAL_TARGETS['real']
    or len(biggan_candidates) < FINAL_TARGETS['biggan_fake']
    or len(sd14_candidates) < FINAL_TARGETS['sd14_fake']
):
    raise RuntimeError(
        'Cross-pool exact duplicate removal left insufficient candidates; '
        'stop and inspect source pools.'
    )

print("Candidate counts after cross-pool deduplication:")
print(f"Real: {len(real_candidates)}")
print(f"BigGAN: {len(biggan_candidates)}")
print(f"Stable Diffusion v1.4: {len(sd14_candidates)}")

rng = random.Random(SEED)

real_selected = rng.sample(
    real_candidates,
    FINAL_TARGETS['real']
)

biggan_selected = rng.sample(
    biggan_candidates,
    FINAL_TARGETS['biggan_fake']
)

sd14_selected = rng.sample(
    sd14_candidates,
    FINAL_TARGETS['sd14_fake']
)

for i, row in enumerate(real_selected):
    row['split'] = (
        'train' if i < 4000
        else 'validation' if i < 5000
        else 'test'
    )

for i, row in enumerate(biggan_selected):
    row['split'] = (
        'train' if i < 5000
        else 'validation'
    )

for row in sd14_selected:
    row['split'] = 'test'

selected = real_selected + biggan_selected + sd14_selected

df_selected = pd.DataFrame(selected)

counts = (
    df_selected
    .groupby(['split', 'generator'])
    .size()
    .unstack(fill_value=0)
)

print("\nFinal dataset distribution:")
print(counts.to_string())

expected = {
    ('train', 'real'): 4000,
    ('train', 'BigGAN'): 5000,

    ('validation', 'real'): 1000,
    ('validation', 'BigGAN'): 1000,

    ('test', 'real'): 1000,
    ('test', 'Stable Diffusion v1.4'): 1000
}

observed = {
    (split, generator): int(count)
    for (split, generator), count
    in df_selected.groupby(['split', 'generator']).size().items()
}

for key, expected_count in expected.items():
    actual_count = observed.get(key, 0)

    assert actual_count == expected_count, (
        f"{key}: expected {expected_count}, got {actual_count}"
    )

assert len(selected) == 13000, (
    f"Expected 13000 samples, got {len(selected)}"
)

hashes = [x['md5'] for x in selected]

assert len(set(hashes)) == len(hashes), (
    'Exact-byte duplicate remains in final selection.'
)

for split in ['train', 'validation', 'test']:

    generators = set(
        x['generator']
        for x in selected
        if x['split'] == split
    )

    if split in {'train', 'validation'}:
        assert 'Stable Diffusion v1.4' not in generators

    if split == 'test':
        assert 'BigGAN' not in generators

print("\n" + "=" * 60)
print("DISTRIBUTION AND EXACT-BYTE UNIQUENESS CHECKS: PASS")
print("=" * 60)

print(f"Total selected samples: {len(selected)}")
print(f"Unique MD5 hashes: {len(set(hashes))}")
print("Train: Real + BigGAN")
print("Validation: Real + BigGAN")
print("Test: Real + Stable Diffusion v1.4")

In [ ]:
if GENIMAGE_OUT.exists():
    shutil.rmtree(GENIMAGE_OUT)

GENIMAGE_OUT.mkdir(parents=True, exist_ok=True)

NORMALIZED_SIZE = (256, 256)
PADDING_COLOR = (127, 127, 127)

def normalize_bytes_to_jpeg(raw_bytes, dest):

    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)

    with Image.open(BytesIO(raw_bytes)) as im:

        rgb = ImageOps.exif_transpose(im).convert('RGB')

        rgb.thumbnail(
            NORMALIZED_SIZE,
            Image.Resampling.LANCZOS
        )

        canvas = Image.new(
            'RGB',
            NORMALIZED_SIZE,
            PADDING_COLOR
        )

        x = (NORMALIZED_SIZE[0] - rgb.width) // 2
        y = (NORMALIZED_SIZE[1] - rgb.height) // 2

        canvas.paste(rgb, (x, y))

        canvas.save(
            dest,
            format='JPEG',
            quality=95,
            subsampling=0,
            optimize=True
        )

export_rows = []

format_ext = {
    'JPEG': '.jpg',
    'PNG': '.png',
    'WEBP': '.webp',
    'BMP': '.bmp',
    'TIFF': '.tiff'
}

for row in selected:

    split = row['split']
    class_name = row['class_name']
    generator = row['generator']

    if class_name == 'real':

        raw_folder = (
            GENIMAGE_OUT / 'raw' / split / 'real'
        )

        norm_folder = (
            GENIMAGE_OUT / 'normalized' / split / 'real'
        )

    elif generator == 'BigGAN':

        raw_folder = (
            GENIMAGE_OUT / 'raw' / split / 'fake' / 'BigGAN'
        )

        norm_folder = (
            GENIMAGE_OUT / 'normalized' / split / 'fake' / 'BigGAN'
        )

    else:

        raw_folder = (
            GENIMAGE_OUT / 'raw' / split / 'fake'
            / 'Stable_Diffusion_v1_4'
        )

        norm_folder = (
            GENIMAGE_OUT / 'normalized' / split / 'fake'
            / 'Stable_Diffusion_v1_4'
        )

    original_format = str(
        row.get('original_format', '')
    ).upper()

    ext = format_ext.get(
        original_format,
        '.bin'
    )

    raw_path = raw_folder / (row['md5'] + ext)

    norm_path = norm_folder / (row['md5'] + '.jpg')

    raw_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    norm_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    raw_path.write_bytes(row['image_bytes'])

    raw_export_hash = hashlib.md5(
        raw_path.read_bytes()
    ).hexdigest()

    if raw_export_hash != row['md5']:

        raise IOError(
            f'Raw export hash mismatch: {raw_path}'
        )

    normalize_bytes_to_jpeg(
        row['image_bytes'],
        norm_path
    )

    with Image.open(norm_path) as im:

        assert im.size == NORMALIZED_SIZE, (
            f'Incorrect image dimensions: {norm_path}'
        )

        assert im.format == 'JPEG', (
            f'Incorrect image format: {norm_path}'
        )

        assert im.mode == 'RGB', (
            f'Incorrect image mode: {norm_path}'
        )

    normalized_md5 = hashlib.md5(
        norm_path.read_bytes()
    ).hexdigest()

    export_record = {
        k: v
        for k, v in row.items()
        if k != 'image_bytes'
    }

    export_record.update({

        'raw_export_path': str(raw_path),

        'normalized_export_path': str(norm_path),

        'normalized_format': 'JPEG',

        'normalized_width': NORMALIZED_SIZE[0],

        'normalized_height': NORMALIZED_SIZE[1],

        'normalized_mode': 'RGB',

        'normalized_md5': normalized_md5

    })

    export_rows.append(export_record)

manifest = pd.DataFrame(export_rows)

manifest_path = GENIMAGE_OUT / 'genimage_manifest.csv'

manifest.to_csv(
    manifest_path,
    index=False
)

for split in ['train', 'validation', 'test']:

    split_manifest = manifest[
        manifest['split'] == split
    ]

    split_manifest.to_csv(
        GENIMAGE_OUT / f'genimage_{split}_manifest.csv',
        index=False
    )

assert len(manifest) == len(selected), (
    'Manifest row count does not match selected dataset.'
)

assert manifest['raw_export_path'].notna().all()

assert manifest['normalized_export_path'].notna().all()

assert manifest['normalized_md5'].notna().all()

assert manifest['raw_export_path'].nunique() == len(manifest)

assert manifest['normalized_export_path'].nunique() == len(manifest)

split_counts = manifest.groupby('split').size()

assert split_counts['train'] == 9000

assert split_counts['validation'] == 2000

assert split_counts['test'] == 2000

print("=" * 60)

print("GENIMAGE EXPORT COMPLETED SUCCESSFULLY")

print("=" * 60)

print(f"Total exported images: {len(manifest)}")

print(f"Raw images: {len(manifest)}")

print(f"Normalized images: {len(manifest)}")

print("\nSplit distribution:")

print(split_counts.to_string())

print("\nGenerator/class distribution:")

print(
    manifest
    .groupby(['split', 'generator'])
    .size()
    .unstack(fill_value=0)
    .to_string()
)

print("\nOutput directory:")

print(GENIMAGE_OUT)

print("\nMaster manifest:")

print(manifest_path)

print("\nAll export and validation checks: PASS")

In [ ]:
format_audit=manifest.groupby(['split','class_name','generator','original_format']).size().reset_index(name='count')
dimension_audit=manifest.groupby(['split','class_name','generator','original_width','original_height']).size().reset_index(name='count')
format_audit.to_csv(REPORTS/'genimage_original_format_audit.csv',index=False)
dimension_audit.to_csv(REPORTS/'genimage_original_dimension_audit.csv',index=False)
print('Original format counts by split/class/generator:')
print(format_audit.to_string(index=False))
print('Original dimension summary by split/class/generator:')
print(manifest.groupby(['split','class_name','generator']).agg(
    image_count=('md5','count'), unique_dimensions=('original_width',lambda x: 'see detail report'),
    min_width=('original_width','min'), max_width=('original_width','max'),
    min_height=('original_height','min'), max_height=('original_height','max')).reset_index().to_string(index=False))

assert set(manifest.normalized_format)=={'JPEG'}
assert set(manifest.normalized_width)=={256}
assert set(manifest.normalized_height)=={256}
assert set(manifest.normalized_mode)=={'RGB'}
print('Standardized format/dimension/mode checks: PASS')

In [ ]:
missing_raw=[]; missing_norm=[]; raw_hash_errors=[]; norm_hash_errors=[]; unreadable=[]; wrong_split_paths=[]
for row in manifest.itertuples():
    rawp=Path(row.raw_export_path); normp=Path(row.normalized_export_path)
    if not rawp.is_file(): missing_raw.append(str(rawp))
    else:
        if hashlib.md5(rawp.read_bytes()).hexdigest()!=row.md5: raw_hash_errors.append(str(rawp))
        try:
            with Image.open(rawp) as im: im.verify()
        except Exception as e: unreadable.append({'path':str(rawp),'error':repr(e)})
    if not normp.is_file(): missing_norm.append(str(normp))
    else:
        if hashlib.md5(normp.read_bytes()).hexdigest()!=row.normalized_md5: norm_hash_errors.append(str(normp))
        try:
            with Image.open(normp) as im:
                im.verify()
            with Image.open(normp) as im:
                if im.size!=(256,256) or im.format!='JPEG' or im.mode!='RGB':
                    unreadable.append({'path':str(normp),'error':'normalized specification mismatch'})
        except Exception as e: unreadable.append({'path':str(normp),'error':repr(e)})
    if f'/{row.split}/' not in rawp.as_posix() or f'/{row.split}/' not in normp.as_posix():
        wrong_split_paths.append({'raw':str(rawp),'normalized':str(normp),'split':row.split})

split_hashes={s:set(manifest.loc[manifest.split==s,'md5']) for s in ['train','validation','test']}
overlaps={'train_validation':len(split_hashes['train']&split_hashes['validation']),
          'train_test':len(split_hashes['train']&split_hashes['test']),
          'validation_test':len(split_hashes['validation']&split_hashes['test'])}

final_counts=manifest.groupby(['split','generator']).size().unstack(fill_value=0).to_dict()
genimage_report={
    'dataset':'GenImage-arrow', 'repo':GENIMAGE_REPO, 'revision':GENIMAGE_REVISION,
    'configs':GENIMAGE_CONFIGS, 'total_records':int(len(manifest)),
    'split_counts':{s:int((manifest.split==s).sum()) for s in ['train','validation','test']},
    'generator_counts':{s:manifest[manifest.split==s].generator.value_counts().to_dict() for s in ['train','validation','test']},
    'unique_source_md5_count':int(manifest.md5.nunique()),
    'missing_raw_paths':len(missing_raw),'missing_normalized_paths':len(missing_norm),
    'raw_hash_errors':len(raw_hash_errors),'normalized_hash_errors':len(norm_hash_errors),
    'unreadable_or_invalid':len(unreadable),'wrong_split_paths':len(wrong_split_paths),
    'exact_hash_overlaps':overlaps,
    'format_dimension_audit_files':['genimage_original_format_audit.csv','genimage_original_dimension_audit.csv'],
    'standardized_spec':{'format':'JPEG','dimensions':[256,256],'mode':'RGB','quality':95},
    'source_byte_status_counts':manifest.source_byte_status.value_counts().to_dict(),
    'limitations':['MD5 detects exact-byte duplicates only, not near-duplicates.',
                  'Square resizing can distort aspect ratio.',
                  'Held-out generator is Stable Diffusion v1.4 only.']
}
with open(REPORTS/'genimage_f1_audit.json','w') as f: json.dump(genimage_report,f,indent=2)

verite_report={
    'dataset':'VERITE','github_repo':VERITE_GITHUB,'github_commit':verite_git_sha,
    'huggingface_repo':VERITE_HF_ID,'huggingface_revision':verite_hf_revision,
    'metadata_rows_total':int(len(verite_check)),
    'rows_resolved_to_image':int(verite_check.image_exists.sum()),
    'rows_missing_or_ambiguous':int((~verite_check.image_exists).sum()),
    'readable_rows_used':int(len(valid_verite)),
    'split_rows':{s:int(len(d)) for s,d in split_dfs.items()},
    'split_unique_groups':{s:int(d.group_id.nunique()) for s,d in split_dfs.items()},
    'group_overlap':{'train_validation':len(group_sets['train']&group_sets['validation']),
                     'train_test':len(group_sets['train']&group_sets['test']),
                     'validation_test':len(group_sets['validation']&group_sets['test'])},
    'normalized_spec':{'format':'JPEG','dimensions':[256,256],'mode':'RGB','quality':95},
    'ambiguous_matches':len(ambiguous_basename_rows),
    'limitations':['Group split uses image_path as group key.',
                   'Square resizing can distort aspect ratio.',
                   'Image availability is limited by gated Hugging Face access.']
}
with open(REPORTS/'verite_f1_audit.json','w') as f: json.dump(verite_report,f,indent=2)

assert len(manifest)==13000
assert manifest.md5.nunique()==len(manifest)
assert all(v==0 for v in overlaps.values())
assert not missing_raw and not missing_norm and not raw_hash_errors and not norm_hash_errors and not unreadable and not wrong_split_paths
assert genimage_report['split_counts']=={'train':9000,'validation':2000,'test':2000}
assert not any(genimage_report['generator_counts'][s].get('Stable Diffusion v1.4',0) for s in ['train','validation'])
assert genimage_report['generator_counts']['test'].get('BigGAN',0)==0
assert all(v==0 for v in verite_report['group_overlap'].values())

summary={'task':'F1 vision data preparation','seed':SEED,'runtime':runtime_info,
         'verite':verite_report,'red_dot_evidence_availability':red_dot_report,
         'genimage':genimage_report,
         'notes':['Raw source images preserved in separate raw folders.',
                  'Standardized JPEG 256x256 RGB copies are used as shortcut-control inputs.',
                  'No model training is performed in F1.',
                  'Exact-byte duplicate audit only; near-duplicate audit is not performed.']}
with open(REPORTS/'F1_summary_report.json','w') as f: json.dump(summary,f,indent=2)
print(json.dumps({'VERITE':verite_report,'RED-DOT':red_dot_report,'GenImage':genimage_report},indent=2))
print('F1 FINAL AUDIT: PASS')

In [ ]:
NOTES_DIR=PROJECT/'notes'
NOTES_DIR.mkdir(parents=True,exist_ok=True)
note=f"""# F1 — Vision data preparation

- **Status:** executed only if final notebook cell prints `F1 FINAL AUDIT: PASS`.
- **VERITE:** {verite_report['readable_rows_used']} readable rows used; split rows {verite_report['split_rows']}; source Git commit `{verite_git_sha}`; HF revision `{verite_hf_revision}`.
- **RED-DOT:** availability check status `{red_dot_report['available']}`; semantic relevance is not assessed.
- **GenImage:** {len(manifest)} images; train/validation/test = 9000/2000/2000; BigGAN train+validation, Stable Diffusion v1.4 test only. Source revision `{GENIMAGE_REVISION}`.
- **Preprocessing:** raw files preserved; standardized copy JPEG, RGB, 256x256, quality 95.
- **Bias audit:** original file format and dimensions are summarized in the F1 reports. Standardization reduces direct file-format/dimension shortcut cues but does not prove all shortcuts are removed.
- **Known limitations:** exact MD5 only (not near-duplicate detection); square resize may distort aspect ratio; held-out generator is only SD v1.4; VERITE HF access is gated.
"""
(NOTES_DIR/'F1.md').write_text(note,encoding='utf-8')
print(note)
print('Reports:',REPORTS)
print('Manifests:',VERITE_OUT,GENIMAGE_OUT)

In [ ]:
PACKAGE_DIR = Path("/content/TwinCheck_F1_Results_Package")

(PACKAGE_DIR / "Results").mkdir(
    parents=True,
    exist_ok=True
)

print("Results folder created successfully.")

In [ ]:
import os
import json
import shutil
import zipfile
import hashlib
from pathlib import Path
from datetime import datetime
import pandas as pd

from google.colab import files

PACKAGE_DIR = Path("/content/TwinCheck_F1_Results_Package")
ZIP_PATH = Path("/content/TwinCheck_F1_Results_Package.zip")

GENIMAGE_OUT = Path(GENIMAGE_OUT)

NOTEBOOK_PATH = None

if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)

PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = PACKAGE_DIR / "Results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ARTIFACTS_DIR = PACKAGE_DIR / "GenImage_Artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

IMPLEMENTATION_DIR = PACKAGE_DIR / "Implementation"
IMPLEMENTATION_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".webp", ".bmp",
    ".tif", ".tiff", ".gif", ".jp2", ".heic"
}

EXCLUDED_FOLDER_NAMES = {
    "raw",
    "normalized",
    "images",
    "images_extracted",
    "image_data"
}

ALLOWED_ARTIFACT_EXTENSIONS = {
    ".csv", ".json", ".txt", ".md", ".py",
    ".ipynb", ".yaml", ".yml", ".log",
    ".html", ".pdf"
}

copied_files = []

if GENIMAGE_OUT.exists():

    for source_file in GENIMAGE_OUT.rglob("*"):

        if not source_file.is_file():
            continue

        if source_file.suffix.lower() in IMAGE_EXTENSIONS:
            continue

        if any(
            folder.lower() in EXCLUDED_FOLDER_NAMES
            for folder in source_file.parts
        ):
            continue

        if source_file.suffix.lower() not in ALLOWED_ARTIFACT_EXTENSIONS:
            continue

        relative_path = source_file.relative_to(GENIMAGE_OUT)

        destination = ARTIFACTS_DIR / relative_path

        destination.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        shutil.copy2(source_file, destination)

        copied_files.append(
            str(destination.relative_to(PACKAGE_DIR))
        )

if NOTEBOOK_PATH is not None:

    NOTEBOOK_PATH = Path(NOTEBOOK_PATH)

    if NOTEBOOK_PATH.exists():

        notebook_destination = (
            IMPLEMENTATION_DIR / NOTEBOOK_PATH.name
        )

        shutil.copy2(
            NOTEBOOK_PATH,
            notebook_destination
        )

        copied_files.append(
            str(notebook_destination.relative_to(PACKAGE_DIR))
        )

    else:

        print("WARNING: Notebook was not found:")
        print(NOTEBOOK_PATH)

else:

    print(
        "NOTE: No notebook was specified. "
        "The implementation notebook will not be included "
        "unless NOTEBOOK_PATH is set."
    )

selected_df = pd.DataFrame(selected)

dataset_summary = (
    selected_df
    .groupby(["split", "class_name", "generator"])
    .size()
    .reset_index(name="count")
)

dataset_summary.to_csv(
    RESULTS_DIR / "final_dataset_distribution.csv",
    index=False
)

split_summary = (
    selected_df
    .groupby("split")
    .size()
    .reset_index(name="total_samples")
)

split_summary.to_csv(
    RESULTS_DIR / "split_totals.csv",
    index=False
)

generator_summary = (
    selected_df
    .groupby("generator")
    .size()
    .reset_index(name="total_samples")
)

generator_summary.to_csv(
    RESULTS_DIR / "generator_totals.csv",
    index=False
)

md5_values = selected_df["md5"].tolist()

quality_results = {
    "stage": "F1 - GenImage data preparation",
    "completion_timestamp": datetime.now().isoformat(),

    "total_selected_samples": int(len(selected)),

    "unique_exact_byte_hashes": int(len(set(md5_values))),

    "exact_byte_duplicates_in_final_selection": int(
        len(md5_values) - len(set(md5_values))
    ),

    "exact_byte_uniqueness_check": (
        len(md5_values) == len(set(md5_values))
    ),

    "expected_total_samples": 13000,

    "total_sample_count_check": (
        len(selected) == 13000
    ),

    "split_counts": {
        str(k): int(v)
        for k, v in selected_df.groupby("split").size().items()
    },

    "generator_counts": {
        str(k): int(v)
        for k, v in selected_df.groupby("generator").size().items()
    },

    "split_generator_class_counts": {
        f"{split}|{class_name}|{generator}": int(count)
        for (split, class_name, generator), count
        in selected_df.groupby(
            ["split", "class_name", "generator"]
        ).size().items()
    }
}

with open(
    RESULTS_DIR / "f1_verification_results.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        quality_results,
        f,
        indent=4
    )

experiment_config = {
    "project": "TwinCheck",
    "stage": "F1",
    "dataset": "GenImage",

    "selected_generators": [
        "Real",
        "BigGAN",
        "Stable Diffusion v1.4"
    ],

    "total_selected_samples": 13000,

    "train": {
        "real": 4000,
        "BigGAN_fake": 5000,
        "total": 9000
    },

    "validation": {
        "real": 1000,
        "BigGAN_fake": 1000,
        "total": 2000
    },

    "test": {
        "real": 1000,
        "Stable_Diffusion_v1_4_fake": 1000,
        "total": 2000
    },

    "normalization": {
        "image_size": [256, 256],
        "format": "JPEG",
        "mode": "RGB",
        "quality": 95,
        "aspect_ratio_preservation": True,
        "padding": True
    },

    "random_seed": int(SEED)
}

with open(
    PACKAGE_DIR / "experiment_configuration.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        experiment_config,
        f,
        indent=4
    )

readme_text = f"""
# TwinCheck - F1 Results & Implementation Package

## Project Stage
F1: GenImage Dataset Preparation

## Purpose
This package contains non-image artifacts for the F1 data
preparation stage of TwinCheck.

Actual image files are intentionally excluded.

## Dataset

Total selected samples: {len(selected)}

### Train
- Real: 4,000
- BigGAN fake: 5,000
- Total: 9,000

### Validation
- Real: 1,000
- BigGAN fake: 1,000
- Total: 2,000

### Test
- Real: 1,000
- Stable Diffusion v1.4 fake: 1,000
- Total: 2,000

## Experimental Design

Training and validation contain Real and BigGAN images.

Testing contains Real and Stable Diffusion v1.4 images.
Stable Diffusion v1.4 is held out from training and validation
for cross-generator generalization evaluation.

## Normalization

- Image size: 256 x 256
- Color mode: RGB
- Format: JPEG
- Aspect ratio preservation: enabled
- Padding: enabled
- JPEG quality: 95

## Package Contents

- GenImage_Artifacts/
    - Non-image dataset artifacts and manifests
- Results/
    - Dataset distribution CSV
    - Split totals CSV
    - Generator totals CSV
    - F1 verification JSON
- Implementation/
    - Notebook, if explicitly included
- experiment_configuration.json
- file_inventory.csv

## Images

Raw and normalized images are excluded.

## Exact-Byte Duplicate Verification

Result: {quality_results["exact_byte_uniqueness_check"]}

## Total Sample Verification

Expected total: 13,000
Actual total: {len(selected)}

Result: {quality_results["total_sample_count_check"]}

## Package Creation Time

{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""

with open(
    PACKAGE_DIR / "README.md",
    "w",
    encoding="utf-8"
) as f:

    f.write(readme_text)

inventory = []

for file_path in PACKAGE_DIR.rglob("*"):

    if file_path.is_file():

        inventory.append({
            "relative_path": str(
                file_path.relative_to(PACKAGE_DIR)
            ),
            "size_bytes": file_path.stat().st_size
        })

inventory_df = pd.DataFrame(inventory)

inventory_df.to_csv(
    PACKAGE_DIR / "file_inventory.csv",
    index=False
)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zipf:

    for file_path in PACKAGE_DIR.rglob("*"):

        if file_path.is_file():

            zipf.write(
                file_path,
                arcname=file_path.relative_to(
                    PACKAGE_DIR.parent
                )
            )

with zipfile.ZipFile(ZIP_PATH, "r") as zipf:

    zip_contents = zipf.namelist()

    image_files_in_zip = [
        name for name in zip_contents
        if Path(name).suffix.lower() in IMAGE_EXTENSIONS
    ]

    assert len(image_files_in_zip) == 0, (
        "Images were unexpectedly included in the ZIP."
    )

    assert len(zip_contents) > 0

print("=" * 65)
print("TWINCHECK F1 PACKAGE CREATED")
print("=" * 65)

print(f"ZIP path: {ZIP_PATH}")

print(
    f"ZIP size: "
    f"{ZIP_PATH.stat().st_size / (1024**2):.2f} MB"
)

print(f"Files included: {len(zip_contents)}")
print(f"Images included: {len(image_files_in_zip)}")

print("\nZIP CONTENTS:")

for name in zip_contents:
    print(" -", name)

print("\nF1 verification:")
print("Total samples:", len(selected))
print("Unique MD5 hashes:", len(set(md5_values)))
print("Exact-byte duplicates:", len(md5_values) - len(set(md5_values)))

print("\nAll package image-exclusion checks: PASS")

print("\nDownloading ZIP...")

files.download(str(ZIP_PATH))